## Start spark session (Mode training)

In [1]:
from scripts import Spark_Init as spi
spark = spi.start_spark("Data Ingestion")

Library location check:
Spark home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\spark-3.5.1-bin-hadoop3
Java home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\java\jdk-17
hadoop home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\hadoop
Spark Session 'Data Ingestion' is active!
Spark UI available at: http://10.223.136.209:4040


In [2]:

import numpy as np
from sklearn.linear_model import LinearRegression as SKLinearRegression
import time
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd

#### Load featured / processed data

In [2]:
df = spark.read.parquet("../data/features_data")

In [4]:
df.printSchema()

root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- distance_per_passenger: double (nullable = true)
 |-- fare_per_distance: double (nullable = true)
 |-- total_per_passenger: double (nullable = true)
 |-- avg_fare_by_location: double (nullable = true)
 |-- avg_duration_by_location: double (nullable = true)
 |-- features: vector (nullable = true)
 |-- scaled_features: vector (nullable = true)



In [5]:
print(f"Training on {df.count()} records.")

Training on 7871432 records.


## Sampling for Scikit-Learn (Simulating single-node constraints)
**Sample to test the single node training in Scikit compared to pyspark**

In [6]:
# Sampling for Scikit-Learn (Simulating single-node constraints)
pdf_sample = df.sample(0.08).toPandas()
X_sk = np.array(pdf_sample['features'].tolist())
y_sk = pdf_sample['fare_amount'].values

start_time = time.time()
sk_model = SKLinearRegression().fit(X_sk, y_sk)
sk_duration = time.time() - start_time

print(f"Scikit-Learn (Single Node) Baseline Training Time: {sk_duration:.2f}s")

Scikit-Learn (Single Node) Baseline Training Time: 0.13s


## Split data for training

In [7]:
df_train, df_test = df.randomSplit([0.8, 0.2], seed=42)
#Cache to improve speed
df_train.cache()

DataFrame[fare_amount: double, trip_distance: double, tip_amount: double, trip_duration: double, distance_per_passenger: double, fare_per_distance: double, total_per_passenger: double, avg_fare_by_location: double, avg_duration_by_location: double, features: vector, scaled_features: vector]

In [8]:
target_col = "fare_amount"
# --- MODEL DEFINITIONS ---
regressors = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol=target_col),
    "DecisionTree": DecisionTreeRegressor(featuresCol="features", labelCol=target_col),
    "RandomForest": RandomForestRegressor(featuresCol="features", labelCol=target_col, numTrees=20),
    "GBT_Regressor": GBTRegressor(featuresCol="features", labelCol=target_col, maxIter=10, maxBins=16)
}

evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
# results = {}
training_log = []

##### --- TRAINING DATASET---

In [9]:
for name, model in regressors.items():
    # --- Time/Memory Analysis ---
    start_time = time.time()

    if name == "GBT_Regressor":
        spark.sparkContext.setCheckpointDir("../data/checkpoints/")
        # Apply Grid Search only for the most complex model
        grid = ParamGridBuilder().addGrid(model.maxDepth, [2, 4, 6]).build()
        cv = CrossValidator(estimator=model, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3, parallelism=2)
        trained_model = cv.fit(df_train).bestModel
    else:
        trained_model = model.fit(df_train)


    training_duration = time.time() - start_time

    # --- Performance Metrics ---
    predictions = trained_model.transform(df_test)
    rmse = evaluator.evaluate(predictions, {evaluator.metricName: "rmse"})
    r2 = evaluator.evaluate(predictions, {evaluator.metricName: "r2"})
    mae = evaluator.evaluate(predictions, {evaluator.metricName: "mae"})

    training_log.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2,
        "MAE": mae,
        "Training_Time_Sec": training_duration,
        "Model_Obj": trained_model,
        "Predictions": predictions.select("fare_amount", "prediction")
    })




#### Create a comparison table

In [10]:
# # --- CHOOSING THE BEST MODEL ---
best_model_dict = min(training_log, key=lambda x:x['RMSE'])

In [11]:
formatted_results = []

for res in training_log:
    formatted_results.append({
        "Model": res["Model"],
        "RMSE": res["RMSE"],
        "R2": res["R2"],
        "MAE": res["MAE"],
        "Training_Time_Sec": res["Training_Time_Sec"]
         })

# Create Comparison Table
results_df = pd.DataFrame(formatted_results)
print("\n### Model Performance Comparison\n")
print(results_df.sort_values(by="RMSE").to_markdown())

print(f"\nWINNER: {best_model_dict['Model']} with RMSE: {best_model_dict['RMSE']:.4f}")


### Model Performance Comparison

|    | Model            |     RMSE |       R2 |      MAE |   Training_Time_Sec |
|---:|:-----------------|---------:|---------:|---------:|--------------------:|
|  3 | GBT_Regressor    | 0.77591  | 0.980594 | 0.569566 |           556.962   |
|  2 | RandomForest     | 0.821532 | 0.978245 | 0.583546 |            39.7041  |
|  0 | LinearRegression | 0.911833 | 0.973199 | 0.772025 |            21.4173  |
|  1 | DecisionTree     | 1.06009  | 0.963775 | 0.771365 |             8.72926 |

WINNER: GBT_Regressor with RMSE: 0.7759


In [18]:
from pyspark.sql.functions import rand

random_predictions = training_log[0]['Predictions'].orderBy(rand()).limit(10)

random_predictions.show()

+-----------+------------------+
|fare_amount|        prediction|
+-----------+------------------+
|       12.1|11.529171580357318|
|       22.6|22.135992851734578|
|       25.4|24.900593793020178|
|       28.2|26.739866333319846|
|       19.1| 18.44735106607971|
|        8.6|7.6401051218322324|
|        7.2|7.5493393787982255|
|       14.9|15.254205739036689|
|       12.1|11.257822548761279|
|       17.7|16.734685164834637|
+-----------+------------------+



###Relative RMSE tells us that the predicted fare amount on an average varies by 7.38% from the actual value

In [21]:
# Calculate the mean of fare_amount in the test set
mean_fare = df_test.select("fare_amount").groupBy().avg().first()[0]

# Calculate relative RMSE as a percentage
relative_rmse = (formatted_results[0]['RMSE'] / mean_fare) * 100
print(f"Relative RMSE: {relative_rmse:.2f}%")

Relative RMSE: 7.10%


### Save trained model and predictions

In [16]:
for log in training_log:
    model_name = log["Model"].lower()

 # Save model (Spark model)
    log["Model_Obj"].write().overwrite().save(f"../data/trained/saved_models/{model_name}_model")

    # Save predictions (Pandas)
    log["Predictions"].write.mode("overwrite").parquet(f"../data/saved_predictions/{model_name}_predictions")